# 04 — Análise: do Gold aos dois achados principais

Este notebook não calcula nada novo. Tudo aqui lê tabelas `gold.bid.*`
produzidas pelo `03_gold_bid_performance` e renderiza as duas comparações
com que o README abre, mais o contexto de qualidade de dados que as
qualifica.

Tanto este notebook quanto o README são gerados a partir da mesma execução
com seed fixa (`--seed 42`), então os percentuais principais devem bater.
Onde uma figura aqui envolve uma decisão de julgamento (ex.: quais motivos
de perda contam como relacionados a preço), a classificação é deixada
explícita na célula em vez de assumida.

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Somente agregados — toda tabela gold aqui é pequena o bastante (um punhado
# de linhas por dimensão) para que um toPandas() de nó único seja a escolha
# certa. Trazer a fact table subjacente pro driver não seria.
overall            = spark.table("gold.bid.performance_overall").toPandas()
by_channel         = spark.table("gold.bid.performance_by_channel").toPandas()
by_executive       = spark.table("gold.bid.performance_by_account_executive").toPandas()
by_segment         = spark.table("gold.bid.performance_by_segment").toPandas()
by_value_band      = spark.table("gold.bid.performance_by_value_band").toPandas()
loss_coverage      = spark.table("gold.bid.loss_reason_coverage").toPandas()
loss_reasons       = spark.table("gold.bid.loss_reasons").toPandas()
open_pipeline      = spark.table("gold.bid.open_pipeline").toPandas()

## Achado 1 — o artefato de migração

Propostas carregadas em lote convertem a aproximadamente um terço da taxa
das registradas organicamente. O gap não é ruído — é 58% da base se
comportando como uma população diferente, concentrada na carteira de um
único executivo.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Painel 1: lote vs orgânico, empresa toda ------------------------------
channel_labels = by_channel["is_bulk_load"].map({True: "Carga em lote", False: "Orgânico"})
axes[0].bar(channel_labels, by_channel["win_rate_by_count"],
            color=["#c0c0c0", "#2b6cb0"])
axes[0].set_title("Taxa de vitória: lote vs orgânico")
axes[0].set_ylabel("Taxa de vitória")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(by_channel["win_rate_by_count"]):
    axes[0].text(i, v + 1, f"{v:.1f}%", ha="center")

# --- Painel 2: Executivo 4, todas as propostas vs somente orgânicas -------
exec4 = by_executive[by_executive["account_executive"] == "Executive 4"].iloc[0]
company_avg = overall["win_rate_by_count"].iloc[0]

bars = axes[1].bar(
    ["Todas as\npropostas", "Somente\norgânicas", "Média da\nempresa"],
    [exec4["wr_all"], exec4["wr_organic"], company_avg],
    color=["#c0392b", "#2b6cb0", "#888888"],
)
axes[1].set_title("Executivo 4: pior desempenho, ou artefato?")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, v in zip(bars, [exec4["wr_all"], exec4["wr_organic"], company_avg]):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + 1, f"{v:.1f}%", ha="center")

plt.tight_layout()
plt.savefig("/tmp/achado1_artefato_migracao.png", bbox_inches="tight")
plt.show()

print(f"Executivo 4 — todas as propostas: {exec4['wr_all']:.1f}%   somente orgânicas: {exec4['wr_organic']:.1f}%   "
      f"gap: {exec4['artefact_gap']:.1f}pp")

### Ressalva que este gráfico não mostra

O lote de carga em massa foi concentrado em Financial Services / Public
Sector — os dois segmentos de menor taxa de vitória de base na base.
Remover a carga em lote move o Executivo 4 de pior desempenho para acima
da média, mas parte do que sobra ainda pode ser mix de segmento, não um
sinal orgânico limpo. A seção logo depois desta testa isso diretamente com
uma regressão que trava segmento, estado e tamanho do negócio como
constantes.

## Controlando o confound: o efeito sobrevive?

O gráfico acima compara propostas carregadas em lote com orgânicas sem
levar em conta *quais* propostas foram parar em cada balde. O gerador
concentrou os registros em lote em Financial Services, Public Sector, e na
carteira do Executivo 4 — segmentos que convertem abaixo da média
independente de como a proposta foi registrada. Então o gap de 16,8pp
poderia ser parcialmente (ou inteiramente) mix de segmento vestido de
artefato de migração.

Uma regressão logística que trava segmento, estado e tamanho do negócio
como constantes responde isso diretamente: depois de ajustar por tudo mais
na linha, o `is_bulk_load` ainda move as chances de vitória?

In [0]:
%pip install statsmodels -q

Se a importação abaixo falhar com um erro de módulo desatualizado, rode
`dbutils.library.restartPython()` numa nova célula e execute o notebook de
novo do início — esta é a primeira célula que importa `statsmodels`, então
normalmente não precisa de restart, mas o cache de ambiente do Databricks é
inconsistente sobre isso.

In [0]:
import numpy as np
import statsmodels.formula.api as smf
from pyspark.sql import functions as F

# Dado no nível de linha, não os agregados do Gold — este é o único lugar
# do notebook que precisa disso. Mesmo join point-in-time do `fact` do
# 03_gold_bid_performance (clients_clean é SCD Type 2 — um join simples por
# client_id faria fan-out das propostas de todo cliente renovado e
# contaria em dobro no modelo).
bids_clean = spark.table("silver.bid.bids_clean")
clients_clean = spark.table("silver.bid.clients_clean")

reg_pdf = (
    bids_clean.alias("b")
    .join(
        clients_clean.alias("c"),
        (F.col("b.client_id") == F.col("c.client_id"))
        & (F.col("b.created_at") >= F.col("c.valid_from").cast("timestamp"))
        & (F.col("c.valid_to").isNull() | (F.col("b.created_at") < F.col("c.valid_to").cast("timestamp"))),
        "left",
    )
    .filter("b.outcome IS NOT NULL")
    .select(
        F.col("b.outcome"), F.col("b.is_bulk_load"),
        F.col("c.segment"), F.col("c.state"), F.col("b.contract_value_brl"),
    )
    .toPandas()
)

# Valor contínuo em log em vez das faixas de quartil da camada Gold — uma
# regressão não precisa de binning, e o log evita que a distribuição de
# valor, assimétrica à direita, domine o ajuste.
reg_pdf["log_value"] = np.log(reg_pdf["contract_value_brl"])

print(f"linhas entrando no modelo: {len(reg_pdf)}")

**Por que segment e state estão no modelo, e account_executive não:**
segment e state são onde o lote de carga em massa foi concentrado — são o
confound. `account_executive` é quase a mesma variável que `is_bulk_load`
aqui (os lotes ficam quase inteiramente na carteira do Executivo 4 por
construção), então incluí-lo absorveria o próprio efeito que este modelo
está tentando medir, em vez de controlar por um confound separado.

In [0]:
model = smf.logit(
    "outcome ~ C(is_bulk_load) + C(segment) + C(state) + log_value",
    data=reg_pdf,
).fit(disp=False)

BULK_TERM = "C(is_bulk_load)[T.True]"
coef = model.params[BULK_TERM]
ci_lo, ci_hi = model.conf_int().loc[BULK_TERM]
pval = model.pvalues[BULK_TERM]

adjusted_or = np.exp(coef)
print(f"Odds ratio ajustado para is_bulk_load: {adjusted_or:.2f}   "
      f"IC 95% [{np.exp(ci_lo):.2f}, {np.exp(ci_hi):.2f}]   p={pval:.1e}")

# Odds ratio ingênuo para comparação — o número que o gráfico do Achado 1
# sugere sem nenhum controle.
p_bulk = reg_pdf.loc[reg_pdf.is_bulk_load, "outcome"].mean()
p_org = reg_pdf.loc[~reg_pdf.is_bulk_load, "outcome"].mean()
naive_or = (p_bulk / (1 - p_bulk)) / (p_org / (1 - p_org))
print(f"Odds ratio sem controles:              {naive_or:.2f}")

**Como ler a comparação:** se o odds ratio ajustado tivesse se movido
fortemente em direção a 1 em relação ao ingênuo, isso significaria que o
mix de segment/state/valor estava fazendo a maior parte do trabalho, e a
história do "artefato de migração" estava superestimada. Se ele fica
próximo do número ingênuo, o efeito da carga em lote é real, não apenas um
proxy de quais segmentos calharam de ser carregados em lote — o que é o
que permite que o número principal do Achado 1 se sustente sem um
asterisco de mix de segmento.

Isso ainda não é prova causal — é um ajuste observacional pelos confounders
que a base torna óbvios (segment, state, tamanho do negócio), não por todo
confounder que poderia existir. `model.summary()` tem a tabela completa de
coeficientes se você quiser ver como segment e state individualmente movem
a probabilidade de vitória.

## Achado 2 — taxa de vitória por contagem vs por valor

A empresa ganha contratos pequenos e perde os grandes. Contar propostas
favorece o desempenho; ponderar por receita não.

In [0]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(by_value_band["value_quartile"].astype(str), by_value_band["win_rate_by_count"],
       color="#2b6cb0")
ax.set_title("Taxa de vitória por quartil de valor de contrato")
ax.set_xlabel("Quartil de valor (1 = menor, 4 = maior)")
ax.set_ylabel("Taxa de vitória")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(by_value_band["win_rate_by_count"]):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center")

plt.tight_layout()
plt.savefig("/tmp/achado2_efeito_valor.png", bbox_inches="tight")
plt.show()

gap = overall["win_rate_by_count"].iloc[0] - overall["win_rate_by_value"].iloc[0]
print(f"Taxa de vitória por contagem: {overall['win_rate_by_count'].iloc[0]:.1f}%   "
      f"por valor: {overall['win_rate_by_value'].iloc[0]:.1f}%   gap: {gap:.1f}pp")

## Por que acontece: motivos de perda, e o quanto pouco cobrem do quadro

O campo de motivo resolveria se o Achado 2 é um problema de precificação.
Ele é preenchido para uma fração de um dígito das perdas — reportado aqui
como um sinal, explicitamente não como uma estimativa populacional.

In [0]:
coverage_pct = loss_coverage["coverage_pct"].iloc[0]
print(f"Cobertura de motivo de perda: {coverage_pct:.1f}% "
      f"({loss_coverage['losses_with_reason'].iloc[0]} de {loss_coverage['losses_total'].iloc[0]} perdas)")
print(f"Atribuídas ao concorrente placeholder: {loss_coverage['placeholder_attributed'].iloc[0]}")

# Lista de exclusão explícita em vez de regex por palavra-chave: com apenas
# um punhado de motivos distintos, nomear os que não são de preço
# diretamente é mais confiável que um match por substring (ex.: "Financial
# proposal not competitive" é sobre preço mas não contém cost/price/commercial).
NON_PRICE_REASONS = {
    "Bid cancelled by client",
    "Scope did not match expectations",
    "Incumbent held established relationship",
    "Bid suspended / postponed",
    "Insufficient information from client",
    "Reason not recorded",
}
price_related = loss_reasons[~loss_reasons["loss_reason"].isin(NON_PRICE_REASONS)]
price_share = price_related["share_pct"].sum()
print(f"Fração dos motivos registrados relacionada a preço/custo: {price_share:.1f}%")

loss_reasons.sort_values("losses", ascending=False).head(10)

## Pipeline em aberto

Reportado isoladamente — nunca dobrado na coluna de perdas, o que
subestimaria a taxa de vitória.

In [0]:
open_pipeline

## O que este notebook não afirma

Veja a seção "O que esta análise não permite afirmar" do README para a
lista completa — duração do ciclo de venda, a amostra não aleatória de
motivo de perda, valor mensal vs total do contrato, e a ausência de custo
de proposta. Repetir aqui seria só duplicação; o ponto é que essas
limitações se aplicam a todo gráfico acima, não somente ao texto ao lado
do qual estão.